# Section 1 — LiveKit Agents

Voice agent built with the `livekit-agents` Python SDK.

- `QuickBiteAgent` is a proper `Agent` subclass with a system persona.
- Two `@function_tool`-decorated methods (`get_order_status`, `cancel_order`).
- Gemini 2.0 Flash as the LLM (via `livekit-plugins-google`).
- STT / TTS are mocked with text I/O since we don't have provider keys in Colab — the LLM + tool-calling logic is real.

**Setup:** add `GOOGLE_API_KEY` to Colab Secrets (key icon on the left sidebar).

In [ ]:
!pip install -q livekit-agents livekit-plugins-google

In [ ]:
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
print('GOOGLE_API_KEY loaded.')

In [ ]:
# --- The Agent (same code as section1_livekit/voice_agent.py) ---
import logging
from livekit.agents import Agent, AgentSession, function_tool, RunContext
from livekit.plugins import google as lk_google

logger = logging.getLogger('quickbite-agent')

ORDER_DB = {
    'ORD-001': {'status': 'delivered', 'eta': None, 'restaurant': 'Pizza Palace', 'items': ['Margherita Pizza', 'Garlic Bread']},
    'ORD-002': {'status': 'in_transit', 'eta': '15 minutes', 'restaurant': 'Burger Barn', 'items': ['Double Cheeseburger', 'Fries']},
    'ORD-003': {'status': 'preparing', 'eta': '30 minutes', 'restaurant': 'Sushi Spot', 'items': ['California Roll', 'Miso Soup']},
    'ORD-004': {'status': 'cancelled', 'eta': None, 'restaurant': 'Taco Town', 'items': ['Burrito Bowl']},
}

SYSTEM_PROMPT = (
    'You are a friendly customer-support assistant for QuickBite, a food delivery app. '
    'You help customers check order status and cancel orders. '
    'Always use the provided tools to look up order information — never guess. '
    'Keep replies short, conversational, and easy to say out loud.'
)

class QuickBiteAgent(Agent):
    def __init__(self) -> None:
        super().__init__(
            instructions=SYSTEM_PROMPT,
            llm=lk_google.LLM(model='gemini-2.0-flash'),
        )

    @function_tool
    async def get_order_status(self, context: RunContext, order_id: str) -> str:
        """Look up the current status of a food delivery order.

        Args:
            order_id: The order identifier (e.g. "ORD-001").
        """
        oid = order_id.strip().upper()
        order = ORDER_DB.get(oid)
        if order is None:
            return f"I couldn't find an order with the ID {oid}. Could you double-check the number?"
        s, r, items = order['status'], order['restaurant'], ', '.join(order['items'])
        if s == 'in_transit':
            return f'Order {oid} from {r} is on the way. ETA is about {order["eta"]}. Items: {items}.'
        if s == 'preparing':
            return f'Order {oid} from {r} is being prepared. It should be ready in about {order["eta"]}. Items: {items}.'
        if s == 'delivered':
            return f'Order {oid} from {r} was already delivered. Items: {items}.'
        if s == 'cancelled':
            return f'Order {oid} has been cancelled.'
        return f'Order {oid} has status: {s}.'

    @function_tool
    async def cancel_order(self, context: RunContext, order_id: str, reason: str) -> str:
        """Cancel a pending food delivery order.

        Args:
            order_id: The order identifier (e.g. "ORD-001").
            reason: The customer's reason for cancelling.
        """
        oid = order_id.strip().upper()
        order = ORDER_DB.get(oid)
        if order is None:
            return f"I couldn't find order {oid}, so there's nothing to cancel."
        if order['status'] == 'delivered':
            return f'Order {oid} has already been delivered, so it can\'t be cancelled.'
        if order['status'] == 'cancelled':
            return f'Order {oid} was already cancelled.'
        ORDER_DB[oid]['status'] = 'cancelled'
        return f'Order {oid} is now cancelled. Reason: {reason}. Refund should show in 3-5 business days.'

print('QuickBiteAgent defined (Agent subclass with 2 @function_tool methods).')

In [ ]:
# --- Demo driver: exercise the Agent's LLM + tools end-to-end ---
# STT/TTS are mocked with text I/O per the spec's allowance.
import json
from livekit.agents.llm import ChatContext

def _parse_args(raw):
    if isinstance(raw, str):
        try: return json.loads(raw)
        except Exception: return {}
    return raw or {}

async def run_turn(agent, chat_ctx, user_text):
    print(f'\n  [user]   "{user_text}"')
    chat_ctx.add_message(role='user', content=user_text)

    stream = agent.llm.chat(chat_ctx=chat_ctx, tools=agent.tools)
    text_parts, tool_calls = [], []
    async for chunk in stream:
        d = getattr(chunk, 'delta', None)
        if d is None: continue
        if getattr(d, 'content', None): text_parts.append(d.content)
        if getattr(d, 'tool_calls', None): tool_calls.extend(d.tool_calls)

    while tool_calls:
        chat_ctx.add_message(role='assistant', content='', tool_calls=tool_calls)
        for tc in tool_calls:
            fn_name = tc.name if hasattr(tc, 'name') else tc.function.name
            raw_args = tc.arguments if hasattr(tc, 'arguments') else tc.function.arguments
            args = raw_args if isinstance(raw_args, dict) else _parse_args(raw_args)
            print(f'  [tool]   {fn_name}({args})')
            fn = getattr(agent, fn_name, None)
            try:
                result = await fn(context=None, **args) if fn else f'Error: no tool named {fn_name}'
            except Exception as e:
                result = f'Tool error: {e}'
            print(f'  [result] {result}')
            chat_ctx.add_message(role='tool', content=result,
                                 tool_call_id=getattr(tc, 'id', None) or getattr(tc, 'call_id', None))

        text_parts, tool_calls = [], []
        stream = agent.llm.chat(chat_ctx=chat_ctx, tools=agent.tools)
        async for chunk in stream:
            d = getattr(chunk, 'delta', None)
            if d is None: continue
            if getattr(d, 'content', None): text_parts.append(d.content)
            if getattr(d, 'tool_calls', None): tool_calls.extend(d.tool_calls)

    reply = ''.join(text_parts).strip()
    if reply:
        chat_ctx.add_message(role='assistant', content=reply)
        print(f'  [agent]  "{reply}"')

print('Demo driver ready.')

In [ ]:
# --- Run the simulated session ---
print('=' * 60)
print('  QuickBite Voice Agent — Simulated Session')
print('  (livekit-agents SDK; Gemini LLM; STT/TTS mocked with text I/O)')
print('=' * 60)

agent = QuickBiteAgent()
chat_ctx = ChatContext()
chat_ctx.add_message(role='system', content=SYSTEM_PROMPT)

turns = [
    'Hey, can you check order ORD-002?',
    'When will it arrive?',
    'Also check ORD-003 for me.',
    'Actually, please cancel ORD-003. I changed my mind about sushi.',
    "One more thing — what's the status of ORD-999?",
]

for i, t in enumerate(turns, 1):
    print(f'\n--- turn {i} ---')
    await run_turn(agent, chat_ctx, t)

print('\n' + '=' * 60)
print('  Session complete. Tool calls are logged above.')
print('=' * 60)

## 1.2 (Bonus) — Swapping STT / TTS providers

Because the LiveKit pipeline is composed from provider plugins, swapping a component is a one-line change on the `AgentSession`. The `QuickBiteAgent` class, its instructions, and its `@function_tool` methods stay identical.

In [ ]:
SWAP_EXAMPLES = '''
# Version A — Deepgram STT + ElevenLabs TTS + Gemini LLM (default)
from livekit.plugins import deepgram, elevenlabs, silero

session = AgentSession(
    stt=deepgram.STT(model="nova-3"),
    tts=elevenlabs.TTS(),
    vad=silero.VAD.load(),
)
await session.start(agent=QuickBiteAgent(), room=ctx.room)

# Version B — swap Deepgram -> OpenAI Whisper (STT) and ElevenLabs -> OpenAI TTS
from livekit.plugins import openai as lk_openai

session = AgentSession(
    stt=lk_openai.STT(model="whisper-1"),        # <-- swapped
    tts=lk_openai.TTS(model="tts-1", voice="nova"),  # <-- swapped
    vad=silero.VAD.load(),
)
await session.start(agent=QuickBiteAgent(), room=ctx.room)

# Version C — swap to Google Cloud STT + Google Cloud TTS
session = AgentSession(
    stt=lk_google.STT(),
    tts=lk_google.TTS(),
    vad=silero.VAD.load(),
)
await session.start(agent=QuickBiteAgent(), room=ctx.room)
'''
print(SWAP_EXAMPLES)